# ДЗ 5. Тензоры: малоранговые представления (CP, Tucker, Tensor-Train)

Продолжение ДЗ 4, но для массивов с тремя и более измерениями. Матричное SVD не обобщается на
тензоры однозначным образом — существует несколько разных, не эквивалентных друг другу способов
обобщить понятие "ранга" и "разложения" на тензоры: **CP** (сумма ранг-1 тензоров, обобщение
SVD), **Tucker** (ядро + факторные матрицы по каждой моде, допускает разный ранг по разным
измерениям) и **Tensor-Train** (цепочка небольших 3-мерных "ядер", радикально не страдает от
проклятия размерности).

Используем библиотеку `tensorly` (`pip install tensorly`, либо уже установлена в Colab) с
PyTorch-бэкендом: `tensorly.set_backend('pytorch')`.

Один из сквозных примеров — **сжатие свёрточного слоя обученной сети через CP-разложение**
(задача 3.1) — это не учебная абстракция, а реальная техника ускорения CNN (Lebedev et al.,
2014), которая пригодится вам через два занятия, когда дойдём до тем про CNN.

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import tensorly as tl
tl.set_backend('pytorch')
torch.manual_seed(0)

## Уровень 1. Базовый

### Задача 1.1. Развёртки тензора (unfolding) и многолинейный ранг

В отличие от матриц, у тензора нет единственного числа "ранг" — вместо этого говорят о
**многолинейном ранге**: наборе рангов $(r_1, \ldots, r_N)$ его развёрток (matricization) по
каждой моде. Развёртка по моде $n$ — это способ "распрямить" тензор в матрицу, взяв ось $n$ как
строки, а всё остальное — как столбцы.

1. Для случайного тензора формы $(4, 5, 6)$ реализуйте (или используйте `tensorly.unfold`)
   развёртку по каждой из 3 мод, убедитесь в правильности формы каждой развёртки.
2. Посчитайте ранг каждой развёртки через `torch.linalg.matrix_rank` — для случайного тензора
   все ранги должны быть максимальны (равны соответствующей размерности).
3. Постройте тензор с **заведомо заниженным** многолинейным рангом (через произведение малого
   "ядра" на факторные матрицы, `torch.einsum('ijk,ai,bj,ck->abc', ...)`) и убедитесь, что ранги
   развёрток действительно оказались меньше.

In [ ]:
# TODO: unfold по каждой моде, ранги развёрток, тензор с заниженным рангом
i, j, k = 4, 5, 6
T_rand = torch.randn(i, j, k)
U0 = tl.unfold(T_rand, 0)
U1 = tl.unfold(T_rand, 1)
U2 = tl.unfold(T_rand, 2)

print("Пункт 1")
print(f"Развёртка 0: форма {U0.shape}")
print(f"Развёртка 1: форма {U1.shape}")
print(f"Развёртка 2: форма {U2.shape}")
print("\nПункт 2")
print(f"Развёртка 0: ранг {torch.linalg.matrix_rank(U0).item()}")
print(f"Развёртка 1: ранг {torch.linalg.matrix_rank(U1).item()}")
print(f"Развёртка 2: ранг {torch.linalg.matrix_rank(U2).item()}")


r1, r2, r3 = 2, 3, 2
G = torch.randn(r1, r2, r3)
A = torch.randn(i, r1)
B = torch.randn(j, r2)
C = torch.randn(k, r3)
T_low = torch.einsum('ijk,ai,bj,ck->abc', G, A, B, C)
U0_low = tl.unfold(T_low, 0)
U1_low = tl.unfold(T_low, 1)
U2_low = tl.unfold(T_low, 2)

print("\nПункт 3")
print(f"Развёртка 0: форма {U0_low.shape}, ранг = {torch.linalg.matrix_rank(U0_low).item()}")
print(f"Развёртка 1: форма {U1_low.shape}, ранг = {torch.linalg.matrix_rank(U1_low).item()}")
print(f"Развёртка 2: форма {U2_low.shape}, ранг = {torch.linalg.matrix_rank(U2_low).item()}")

Пункт 1
Развёртка 0: форма torch.Size([4, 30])
Развёртка 1: форма torch.Size([5, 24])
Развёртка 2: форма torch.Size([6, 20])

Пункт 2
Развёртка 0: ранг 4
Развёртка 1: ранг 5
Развёртка 2: ранг 6

Пункт 3
Развёртка 0: форма torch.Size([4, 30]), ранг = 2
Развёртка 1: форма torch.Size([5, 24]), ранг = 3
Развёртка 2: форма torch.Size([6, 20]), ранг = 2


### Задача 1.2. разложение Таккера

Tucker-разложение представляет тензор как **ядро** меньшего размера, "раздутое" факторными
матрицами по каждой моде: $T \approx \mathcal{G} \times_1 A \times_2 B \times_3 C$. В отличие от
CP, ранги по разным модам **независимы** — прямое обобщение многолинейного ранга из задачи 1.1.

1. Постройте тензор с известными Tucker-рангами (например, `(3, 4, 2)`, разными по каждой моде!).
2. Разложите его функцией `tensorly.decomposition.tucker` с разными наборами рангов: заниженным,
   точным и завышенным.
3. Сравните с CP-разложением того же тензора (задача 1.2) по числу параметров при сопоставимой
   ошибке восстановления — какой метод компактнее для этого конкретного тензора?

In [15]:
# TODO: Tucker-разложение того же тензора, сравнение с CP
from tensorly.decomposition import tucker, parafac

I, J, K = 20, 25, 15
r1, r2, r3 = 3, 4, 2
G_true = torch.randn(r1, r2, r3)
A_true = torch.randn(I, r1)
B_true = torch.randn(J, r2)
C_true = torch.randn(K, r3)
T = tl.tenalg.multi_mode_dot(G_true, [A_true, B_true, C_true])

ranks_under = (2, 3, 1)
ranks_exact = (r1, r2, r3)
ranks_over = (5, 6, 4)

for ranks in [ranks_under, ranks_exact, ranks_over]:
    core, factors = tucker(T, rank=ranks)
    T_approx = tl.tenalg.multi_mode_dot(core, factors)
    error = torch.norm(T - T_approx).item() / torch.norm(T).item()
    
    n_params = 1
    for r in ranks:
        n_params *= r
    n_params += I * ranks[0] + J * ranks[1] + K * ranks[2]
    
    print(f"Ranks {ranks}: ошибка = {error:.2e}, параметров = {n_params}")

print()

cp_rank = 10
cp_factors = parafac(T, rank=cp_rank)
T_cp = tl.cp_to_tensor(cp_factors)
error_cp = torch.norm(T - T_cp).item() / torch.norm(T).item()
n_params_cp = cp_rank * (I + J + K)

print(f"CP rank {cp_rank}: ошибка = {error_cp:.2e}, параметров = {n_params_cp}")

Ranks (2, 3, 1): ошибка = 3.55e-01, параметров = 136
Ranks (3, 4, 2): ошибка = 4.92e-07, параметров = 214
Ranks (5, 6, 4): ошибка = 3.73e-07, параметров = 430

CP rank 10: ошибка = 8.86e-07, параметров = 600


## Уровень 2. Посложнее

### Задача 2.1. Сжатие "цветного изображения" через Tucker с разными рангами по модам

Цветное изображение — это естественный тензор 3-го порядка: `(высота, ширина, 3 канала)`.
Используйте Tucker с **сильно асимметричными** рангами: большими для пространственных измерений,
но не больше 3 для канала цвета (больше и не нужно — там ровно 3 канала).

1. Постройте (или сгенерируйте синтетически) изображение как тензор.
2. Сожмите через Tucker с рангами вида `(r, r, 3)` для нескольких `r`.
3. Постройте кривую "ошибка от степени сжатия" и визуализируйте несколько результатов.

In [ ]:
# TODO: Tucker-сжатие изображения-тензора с рангами (r,r,3)

### Задача 2.2. Tensor-Train и проклятие размерности

Полный тензор порядка $N$ с размером моды $d$ занимает $d^N$ элементов — при $N=20$, $d=8$ это
$\sim 10^{18}$ чисел, физически невозможно хранить. Tensor-Train представляет тензор как цепочку
малых 3-мерных "ядер" $G_1, \ldots, G_N$ (каждое — $(r_{n-1}, d, r_n)$), и число параметров растёт
**линейно** по $N$, а не экспоненциально.

1. Постройте тензор 6-го порядка (размер моды 8 по каждой из 6 осей) с известным TT-рангом,
   явно перемножив TT-ядра.
2. Разложите его функцией `tensorly.decomposition.tensor_train` с разными рангами, сравните
   ошибку восстановления и число параметров с полным размером тензора.
3. Постройте (аналитически, без реального построения тензора — это невозможно уже при небольшом
   числе мод!) таблицу/график: как растёт число элементов полного тензора и число параметров
   TT-разложения (при фиксированном ранге) с увеличением числа мод от 2 до 20.

In [ ]:
# TODO: TT-разложение тензора 6-го порядка + таблица роста параметров

---
**Что сдавать:** ноутбук с реализованными функциями, графиками и письменными ответами на вопросы
для решённых задач. Требуется `pip install tensorly` (в Colab одна команда, без дополнительной
настройки). Все вычисления в сумме — не больше нескольких минут на CPU.